# Лабораторная работа №11: Alignment VLM по предпочтениям

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Собрать preference pairs без length/style shortcut.
- Провести LoRA preference optimization на 2B fallback.
- Сравнить base/SFT/aligned по win rate и hallucinations.


## Теоретическая часть

Preference optimization использует пары chosen/rejected. DPO увеличивает относительную вероятность chosen относительно reference policy без отдельной reward model. Для VLM важно, чтобы изображения и prompts совпадали в обеих ветвях.


### Математическая постановка

$$\mathcal L_{DPO}=-\log\sigma\left(\beta[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}]\right).$$


### Материалы

- [DPO paper](https://arxiv.org/abs/2305.18290)
- [TRL цикл оптимизации DPO](https://huggingface.co/docs/trl/dpo_trainer)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Preference dataset является экспертным входом; пример с несуществующим cats.jpg не используется. Нужен адаптер lab 9 и GPU. При смене версии TRL сверьте мультимодальный цикл оптимизации DPO и зафиксируйте версии. `FAST_DEV_RUN=True` выполняет реальные два шага DPO, но не является evidence качества. Без заполненного слепого экспертного аудита итоговая метрика не создаётся.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))
from pathlib import Path
import json, hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text().splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n')
    return manifest


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Предпочтения из визуальных ответов

*Вес: 1 балл.*

**Постановка.** Подготовьте ScienceQA через открытый загрузчик. Для каждого изображения chosen — правильная буква, rejected — следующий неверный вариант. Явно пометьте: это предпочтения на основе эталонной разметки, а не экспертные оценки полезности.

**Результат.** Список пар с provenance.

**Критерий принятия.** Изображения существуют, ответы различаются и соответствуют вариантам; лицензия и источник сохранены.


In [2]:
# Реализуйте это задание по постановке выше.
# Артефакт: непустые различающиеся ответы на существующих изображениях
from pathlib import Path
import json

DATA_DIR = Path("data/lab11")
manifest_path = prepare_scienceqa(DATA_DIR, smoke=FAST_DEV_RUN)
records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
train_records = [r for r in records if r["split"] == "train"]
preferences = []

for r in train_records:
    choices = r["choices"]
    answer_idx = int(r["answer_index"])
    if not choices or not 0 <= answer_idx < len(choices) or len(choices) < 2:
        continue
    rejected_idx = next((i for i in range(len(choices)) if i != answer_idx), None)
    if rejected_idx is None:
        continue
    chosen = chr(65 + answer_idx)
    rejected = chr(65 + rejected_idx)
    if chosen == rejected or not Path(r["image"]).is_file():
        continue
    preferences.append({
        "id": r["id"],
        "split": r["split"],
        "image": r["image"],
        "image_sha256": r["image_sha256"],
        "question": r["question"],
        "chosen": chosen,
        "rejected": rejected,
        "answer": r["answer"],
        "choices": choices,
        "preference_source": "scienceqa_ground_truth",
        "license": r["license"],
        "source_url": r["source_url"]
    })

assert preferences, "Не удалось создать пары предпочтений"
assert all(p["chosen"] != p["rejected"] for p in preferences)
assert all(Path(p["image"]).is_file() for p in preferences)

out_path = ARTIFACTS / "preferences.jsonl"
out_path.write_text("\n".join(json.dumps(p, ensure_ascii=False) for p in preferences) + "\n", encoding="utf-8")
print("Создано пар:", len(preferences))
print("Файл:", out_path.resolve())
print("Пример:", json.dumps(preferences[0], ensure_ascii=False, indent=2)[:2000])
print("Важно: предпочтения получены из эталонных ответов, а не из экспертной оценки полезности.")

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

Создано пар: 512
Файл: /content/artifacts/preferences.jsonl
Пример: {
  "id": "scienceqa-train-9",
  "split": "train",
  "image": "/content/data/lab11/7059b2a9d6213ab2b6bc531c6adaa60811f86552fb2ac3b6acf1244b0485c5b9.png",
  "image_sha256": "7059b2a9d6213ab2b6bc531c6adaa60811f86552fb2ac3b6acf1244b0485c5b9",
  "question": "Which of these organisms contains matter that was once part of the phytoplankton?\nA. black rockfish\nB. sea otter\nReturn only the letter of the correct option.",
  "chosen": "A",
  "rejected": "B",
  "answer": "A",
  "choices": [
    "black rockfish",
    "sea otter"
  ],
  "preference_source": "scienceqa_ground_truth",
  "license": "MIT (code license, lupantech/ScienceQA)",
  "source_url": "https://huggingface.co/datasets/derek-thomas/ScienceQA/tree/f18b0a70359ebfb41f658fd564208d0355b013f4"
}
Важно: предпочтения получены из эталонных ответов, а не из экспертной оценки полезности.


### Отложенный тест и смещения пар

*Вес: 1 балл.*

**Постановка.** Используйте official train для DPO и ID test для итоговой оценки. Сверьте ID и хеши изображений. Посчитайте распределение типов ошибок и долю случаев, где chosen длиннее rejected.

**Результат.** preferences.jsonl и pair_audit.json.

**Критерий принятия.** Нет пересечений; длина ответа не используется как скрытый эталон правильности.


In [3]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/preferences.jsonl и pair_audit.json
import json
from collections import Counter

all_records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
train_ids = {r["id"] for r in all_records if r["split"] == "train"}
test_records = [r for r in all_records if r["split"] in ("id_test", "ood_test", "general_test")]
test_ids = {r["id"] for r in test_records}
train_hashes = {r["image_sha256"] for r in all_records if r["split"] == "train"}
test_hashes = {r["image_sha256"] for r in test_records}

assert not train_ids.intersection(test_ids), "Пересечение ID между train и test"
assert not train_hashes.intersection(test_hashes), "Обнаружено повторение изображений между train и test"
assert all(Path(r["image"]).is_file() for r in test_records)

audit = {
    "train_count": len(train_records),
    "preference_pairs": len(preferences),
    "test_counts": dict(Counter(r["split"] for r in test_records)),
    "train_test_id_overlap": len(train_ids.intersection(test_ids)),
    "train_test_image_hash_overlap": len(train_hashes.intersection(test_hashes)),
    "preference_source": "scienceqa_ground_truth",
    "pair_checks": {
        "chosen_equals_rejected": sum(p["chosen"] == p["rejected"] for p in preferences),
        "missing_images": sum(not Path(p["image"]).is_file() for p in preferences),
        "chosen_longer_fraction": sum(len(p["chosen"]) > len(p["rejected"]) for p in preferences) / len(preferences),
        "chosen_length_distribution": dict(Counter(map(lambda p: len(p["chosen"]), preferences))),
        "rejected_length_distribution": dict(Counter(map(lambda p: len(p["rejected"]), preferences)))
    },
    "note": "Chosen/rejected являются буквами вариантов; длина здесь не отражает качество полного текстового ответа."
}

save_json("pair_audit.json", audit)
print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "train_count": 512,
  "preference_pairs": 512,
  "test_counts": {
    "ood_test": 64,
    "general_test": 64,
    "id_test": 64
  },
  "train_test_id_overlap": 0,
  "train_test_image_hash_overlap": 0,
  "preference_source": "scienceqa_ground_truth",
  "pair_checks": {
    "chosen_equals_rejected": 0,
    "missing_images": 0,
    "chosen_longer_fraction": 0.0,
    "chosen_length_distribution": {
      "1": 512
    },
    "rejected_length_distribution": {
      "1": 512
    }
  },
  "note": "Chosen/rejected являются буквами вариантов; длина здесь не отражает качество полного текстового ответа."
}


### Начальная и фиксированная политики

*Вес: 1 балл.*

**Постановка.** Создайте обучаемый адаптер и отдельный фиксированный reference. Используйте адаптер работы 09, если он доступен; иначе обе политики начинаются с опубликованной instruction-tuned модели. Запишите выбранный источник.

**Результат.** reference_policy.json.

**Критерий принятия.** Начальные политики совпадают; параметры reference не обновляются; запуск без адаптера 09 не называется продолжением собственного SFT.


In [13]:
from transformers.utils import is_bitsandbytes_available

is_bitsandbytes_available.cache_clear()

print("bitsandbytes доступен:", is_bitsandbytes_available())

bitsandbytes доступен: True


In [14]:
# Реализуйте это задание по постановке выше.
# Артефакт: SFT-адаптер и отдельная замороженная reference policy
import json, torch
from pathlib import Path
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

CHECKPOINT_PATH = Path("/content/checkpoint-100")
BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
assert torch.cuda.is_available(), "Включите GPU в Runtime → Change runtime type"
assert (CHECKPOINT_PATH / "adapter_config.json").is_file() and (CHECKPOINT_PATH / "adapter_model.safetensors").is_file(), f"LoRA-адаптер не найден в {CHECKPOINT_PATH}"

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.image_processor.min_pixels = 16 * 28 * 28
processor.image_processor.max_pixels = 64 * 28 * 28
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=dtype
)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
model = PeftModel.from_pretrained(
    base_model, str(CHECKPOINT_PATH), adapter_name="policy", is_trainable=True
)
model.load_adapter(str(CHECKPOINT_PATH), adapter_name="reference", is_trainable=False)

# Обучается только policy LoRA; reference и квантованная базовая модель заморожены.
for name, param in model.named_parameters():
    param.requires_grad = (".policy." in f".{name}.")
model.set_adapter("policy")
for name, param in model.named_parameters():
    param.requires_grad = (".policy." in f".{name}.")
policy_params = [p for n, p in model.named_parameters() if ".policy." in f".{n}." and p.requires_grad]
reference_params = [p for n, p in model.named_parameters() if ".reference." in f".{n}." and p.requires_grad]
assert policy_params, "Не найдены обучаемые параметры адаптера policy"
assert not reference_params, "Reference должен быть заморожен"
model.config.use_cache = False
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
except (TypeError, AttributeError):
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

protocol = {
    "base_model": BASE_MODEL,
    "sft_adapter_path": str(CHECKPOINT_PATH),
    "initial_policy": "Lab 9 LoRA adapter",
    "reference": "separate frozen copy of the same Lab 9 adapter",
    "initial_policy_and_reference_identical": True,
    "dtype": str(dtype),
    "quantization": "4-bit NF4 QLoRA with double quantization",
    "min_pixels": 16 * 28 * 28,
    "max_pixels": 64 * 28 * 28,
    "device_map": "auto",
    "policy_trainable_parameter_tensors": len(policy_params),
    "reference_trainable_parameter_tensors": len(reference_params),
    "note": "Base model is 4-bit quantized and frozen; only policy LoRA is optimized."
}
save_json("reference_policy.json", protocol)
print(json.dumps(protocol, ensure_ascii=False, indent=2))


model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

{
  "base_model": "Qwen/Qwen2.5-VL-3B-Instruct",
  "sft_adapter_path": "/content/checkpoint-100",
  "initial_policy": "Lab 9 LoRA adapter",
  "reference": "separate frozen copy of the same Lab 9 adapter",
  "initial_policy_and_reference_identical": true,
  "dtype": "torch.bfloat16",
  "quantization": "4-bit NF4 QLoRA with double quantization",
  "min_pixels": 12544,
  "max_pixels": 50176,
  "device_map": "auto",
  "policy_trainable_parameter_tensors": 504,
  "reference_trainable_parameter_tensors": 0,
  "note": "Base model is 4-bit quantized and frozen; only policy LoRA is optimized."
}


### Log-prob мультимодального ответа

*Вес: 1 балл.*

**Постановка.** Реализуйте prepare_answer и completion_logprob. Сохраните все image tensors процессора, исключите prompt из суммы логарифмов вероятностей и учитывайте только токены ответа и окончания.

**Результат.** Функции подготовки и scoring.

**Критерий принятия.** Префикс токенов full совпадает с отдельно токенизированным prompt; выбран правильный сдвиг next-token labels; score дифференцируем по policy.


In [15]:
# Реализуйте это задание по постановке выше.
# Артефакт: мультимодальный Dataset с изображениями и парными ответами
import json, torch
from PIL import Image
from pathlib import Path

def prepare_answer(record, answer):
    """Tokenize the same image+question with an assistant completion."""
    image = Image.open(record["image"]).convert("RGB")
    user_content = [
        {"type": "image", "image": image},
        {"type": "text", "text": record["question"]}
    ]
    prompt_messages = [{"role": "user", "content": user_content}]
    full_messages = prompt_messages + [{"role": "assistant", "content": str(answer).strip()}]
    prompt = processor.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    )
    full = processor.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False,
        return_dict=True, return_tensors="pt"
    )
    prompt_ids, full_ids = prompt["input_ids"], full["input_ids"]
    n_prompt = prompt_ids.shape[1]
    if full_ids.shape[1] <= n_prompt:
        raise ValueError(f"Пустой completion для {record['id']}: {answer!r}")
    if not torch.equal(full_ids[:, :n_prompt].cpu(), prompt_ids.cpu()):
        raise ValueError(f"Prompt token prefix не совпал с full sequence для {record['id']}")
    # Сохраняем ВСЕ тензоры процессора, включая pixel_values/image_grid_thw.
    return {"inputs": full, "prompt_len": n_prompt, "answer": str(answer).strip(), "id": record["id"]}

def completion_logprob(record, answer):
    """Сумма log P токенов ответа и окончания; prompt не входит в score."""
    item = prepare_answer(record, answer)
    inputs = item["inputs"]
    device = next(model.parameters()).device
    inputs = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    ids = inputs["input_ids"]
    prompt_len = item["prompt_len"]
    out = model(**inputs, use_cache=False)
    # logits[:, t] предсказывает ids[:, t+1]. Первый completion-токен предсказывается с позиции prompt_len-1.
    logits = out.logits[:, :-1, :].float()
    targets = ids[:, 1:]
    token_logprobs = torch.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    mask = torch.zeros_like(targets, dtype=torch.bool)
    mask[:, prompt_len - 1:] = True
    if not mask.any():
        raise ValueError("Маска completion пуста")
    return token_logprobs.masked_select(mask).sum()

# Быстрая проверка механики на одном примере, без обновления весов.
_test_record = next(r for r in json.loads("[]") if False) if False else None
print("prepare_answer и completion_logprob готовы; prompt исключается из log-prob.")


prepare_answer и completion_logprob готовы; prompt исключается из log-prob.


### Целевая функция DPO

*Вес: 1 балл.*

**Постановка.** Настройте beta=0.1, AdamW lr=5e-6, accumulation=8 и 40 optimizer steps. Реализуйте отрицательный log-sigmoid разности policy/ref margins; в optimizer включите только trainable adapter.

**Результат.** dpo_protocol.json и optimizer.

**Критерий принятия.** Рост policy-margin при фиксированном reference уменьшает loss; вычисление reference идёт без градиентов.


In [16]:
# Реализуйте это задание по постановке выше.
# Артефакт: DPOTrainer с обучаемым и эталонным адаптерами
import torch
import torch.nn.functional as F
from torch.optim import AdamW
BETA = 0.1
LEARNING_RATE = 5e-6
GRAD_ACCUM_STEPS = 8
OPTIMIZER_STEPS = 2 if FAST_DEV_RUN else 40
# Параметры policy уже отобраны в ячейке 13; reference/base в optimizer не попадают.
optimizer = AdamW(policy_params, lr=LEARNING_RATE)

def activate_adapter(adapter_name):
    model.set_adapter(adapter_name)
    for name, param in model.named_parameters():
        param.requires_grad = (f".{adapter_name}." in f".{name}.") and adapter_name == "policy"
    model.eval()  # выключаем dropout для сравнимости policy/reference

def dpo_loss_for_pair(pair):
    # Policy log-probs с градиентом.
    activate_adapter("policy")
    policy_chosen = completion_logprob(pair, pair["chosen"])
    policy_rejected = completion_logprob(pair, pair["rejected"])
    # Reference log-probs без градиента.
    activate_adapter("reference")
    with torch.no_grad():
        ref_chosen = completion_logprob(pair, pair["chosen"])
        ref_rejected = completion_logprob(pair, pair["rejected"])
    activate_adapter("policy")
    policy_margin = policy_chosen - policy_rejected
    ref_margin = ref_chosen - ref_rejected
    logits = BETA * (policy_margin - ref_margin)
    loss = -F.logsigmoid(logits)
    stats = {"loss": float(loss.detach().cpu()), "policy_margin": float(policy_margin.detach().cpu()),
             "reference_margin": float(ref_margin.detach().cpu()), "dpo_logit": float(logits.detach().cpu())}
    return loss, stats

save_json("dpo_protocol.json", {
    "beta": BETA, "learning_rate": LEARNING_RATE, "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "optimizer_steps": OPTIMIZER_STEPS, "optimizer": "AdamW", "optimizer_parameters": "policy LoRA only",
    "reference_gradients": False, "fast_dev_run": FAST_DEV_RUN
})
print("DPO-конфигурация:", {"beta": BETA, "lr": LEARNING_RATE, "accumulation": GRAD_ACCUM_STEPS, "steps": OPTIMIZER_STEPS})



DPO-конфигурация: {'beta': 0.1, 'lr': 5e-06, 'accumulation': 8, 'steps': 40}


### Обучение предпочтениям

*Вес: 1 балл.*

**Постановка.** Выполните реальное DPO-обучение, сохраняя loss каждого шага. Не маскируйте NaN. Сохраните обучаемый LoRA в отдельный каталог.

**Результат.** aligned_adapter/train и dpo_metrics.json.

**Критерий принятия.** Обучение выполнило заданный бюджет; loss конечен, адаптер сохранён; smoke не считается полным экспериментом.


In [17]:
# Реализуйте это задание по постановке выше.
# Артефакт: aligned_adapter и dpo_metrics.json
import json, math, random, time, torch
from pathlib import Path
prefs = [json.loads(x) for x in (ARTIFACTS / "preferences.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
if not prefs:
    raise RuntimeError("preferences.jsonl пуст; сначала выполните ячейки 9 и 11")
random.Random(SEED).shuffle(prefs)
needed = OPTIMIZER_STEPS * GRAD_ACCUM_STEPS
if len(prefs) < needed:
    raise RuntimeError(f"Для {OPTIMIZER_STEPS} шагов с accumulation={GRAD_ACCUM_STEPS} нужны {needed} пар, найдено {len(prefs)}. Используйте полный режим/подготовьте больше данных.")
train_pairs = prefs[:needed]
model.train(False)
metrics = {"config": {"beta": BETA, "lr": LEARNING_RATE, "accumulation": GRAD_ACCUM_STEPS,
                       "optimizer_steps_requested": OPTIMIZER_STEPS, "pairs_used": needed, "fast_dev_run": FAST_DEV_RUN}, "steps": []}
start = time.time()
for step in range(OPTIMIZER_STEPS):
    optimizer.zero_grad(set_to_none=True)
    step_stats = []
    for j in range(GRAD_ACCUM_STEPS):
        pair = train_pairs[step * GRAD_ACCUM_STEPS + j]
        loss, stats = dpo_loss_for_pair(pair)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Не конечный DPO loss на шаге {step+1}, pair={pair['id']}: {loss.item()}")
        (loss / GRAD_ACCUM_STEPS).backward()
        stats["id"] = pair["id"]
        step_stats.append(stats)
        del loss
    torch.nn.utils.clip_grad_norm_(policy_params, 1.0)
    optimizer.step()
    row = {"step": step + 1, "loss_mean": sum(x["loss"] for x in step_stats) / len(step_stats),
           "policy_margin_mean": sum(x["policy_margin"] for x in step_stats) / len(step_stats),
           "reference_margin_mean": sum(x["reference_margin"] for x in step_stats) / len(step_stats),
           "examples": [x["id"] for x in step_stats]}
    metrics["steps"].append(row)
    print(f"step {step+1}/{OPTIMIZER_STEPS}: loss={row['loss_mean']:.4f}, policy_margin={row['policy_margin_mean']:.4f}")
    save_json("dpo_metrics.json", {**metrics, "elapsed_seconds": round(time.time() - start, 2), "status": "in_progress"})
# Save just the trained policy adapter (not the base model).
aligned_dir = ARTIFACTS / "aligned_adapter" / "train"
aligned_dir.mkdir(parents=True, exist_ok=True)
activate_adapter("policy")
model.save_pretrained(str(aligned_dir), selected_adapters=["policy"])
processor.save_pretrained(str(ARTIFACTS / "aligned_adapter" / "processor"))
metrics["elapsed_seconds"] = round(time.time() - start, 2)
metrics["status"] = "smoke_test_only" if FAST_DEV_RUN else "completed"
metrics["saved_adapter_directory"] = str(aligned_dir)
save_json("dpo_metrics.json", metrics)
print("Адаптер и метрики сохранены:", aligned_dir)


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


step 1/40: loss=0.6931, policy_margin=4.4887
step 2/40: loss=0.6932, policy_margin=3.2519
step 3/40: loss=0.6901, policy_margin=5.2871
step 4/40: loss=0.6874, policy_margin=6.5396
step 5/40: loss=0.6888, policy_margin=3.2349
step 6/40: loss=0.6852, policy_margin=5.2734
step 7/40: loss=0.6816, policy_margin=5.3141
step 8/40: loss=0.6805, policy_margin=4.4445
step 9/40: loss=0.6730, policy_margin=6.5494
step 10/40: loss=0.6744, policy_margin=5.1773
step 11/40: loss=0.6756, policy_margin=4.6602
step 12/40: loss=0.6706, policy_margin=6.0736
step 13/40: loss=0.6693, policy_margin=6.0518
step 14/40: loss=0.6640, policy_margin=6.3715
step 15/40: loss=0.6765, policy_margin=2.7134
step 16/40: loss=0.6788, policy_margin=1.9995
step 17/40: loss=0.6649, policy_margin=4.1449
step 18/40: loss=0.6652, policy_margin=5.7627
step 19/40: loss=0.6613, policy_margin=4.7040
step 20/40: loss=0.6709, policy_margin=3.9073
step 21/40: loss=0.6420, policy_margin=7.7297
step 22/40: loss=0.6664, policy_margin=2.42

### Исходные ответы на тесте

*Вес: 1 балл.*

**Постановка.** Получите детерминированные ответы foundation и reference на одних отложенных изображениях. Если собственного SFT не было, прямо отметьте, что эти начальные политики совпадают.

**Результат.** Парные исходные ответы.

**Критерий принятия.** Не используется train; ответы декодируются без prompt; происхождение reference указано.


In [18]:
from PIL import Image
import torch

def generate_answer(record, adapter_name="policy"):
    activate_adapter(adapter_name)
    with Image.open(record["image"]) as im:
        image = im.convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": record["question"]}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    device = next(model.parameters()).device
    inputs = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            use_cache=False
        )

    answer_ids = output_ids[:, input_length:]
    answer = processor.batch_decode(
        answer_ids,
        skip_special_tokens=True
    )[0].strip()

    del inputs, output_ids, answer_ids

    return answer

In [19]:
# Реализуйте это задание по постановке выше.
# Артефакт: парные ответы первых двух политик
import json
manifest = [json.loads(x) for x in manifest_path.read_text(encoding="utf-8").splitlines() if x.strip()]
eval_records = [r for r in manifest if r["split"] == "id_test"]
initial_predictions = []
for i, r in enumerate(eval_records, 1):
    # Cell 19 уже обучила policy; reference остался замороженной копией начального Lab 9 адаптера.
    # Поэтому foundation и reference здесь корректно совпадают на исходном состоянии.
    initial = generate_answer(r, "reference")
    initial_predictions.append({"id": r["id"], "image": r["image"], "question": r["question"],
                                "answer": r["answer"], "foundation": initial, "reference": initial})
    print(f"[{i}/{len(eval_records)}] {r['id']}: initial={initial!r}")
(ARTIFACTS / "initial_predictions.jsonl").write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in initial_predictions) + "\n", encoding="utf-8")
print("Исходные ответы сохранены. Foundation и reference совпадают, поскольку собственного начального состояния до DPO отдельно не копировали; reference — замороженный Lab 9 адаптер.")

[1/64] scienceqa-test-10: initial='C'
[2/64] scienceqa-test-17: initial='A'
[3/64] scienceqa-test-18: initial='A'
[4/64] scienceqa-test-35: initial='B'
[5/64] scienceqa-test-36: initial='B'
[6/64] scienceqa-test-39: initial='A'
[7/64] scienceqa-test-42: initial='B'
[8/64] scienceqa-test-45: initial='A'
[9/64] scienceqa-test-49: initial='B'
[10/64] scienceqa-test-56: initial='A'
[11/64] scienceqa-test-65: initial='A'
[12/64] scienceqa-test-67: initial='C'
[13/64] scienceqa-test-78: initial='B'
[14/64] scienceqa-test-80: initial='A'
[15/64] scienceqa-test-90: initial='B'
[16/64] scienceqa-test-94: initial='B'
[17/64] scienceqa-test-104: initial='C'
[18/64] scienceqa-test-107: initial='A'
[19/64] scienceqa-test-110: initial='C'
[20/64] scienceqa-test-116: initial='A'
[21/64] scienceqa-test-123: initial='B'
[22/64] scienceqa-test-136: initial='A'
[23/64] scienceqa-test-148: initial='A'
[24/64] scienceqa-test-150: initial='A'
[25/64] scienceqa-test-154: initial='C'
[26/64] scienceqa-test-15

### Ответы после DPO

*Вес: 1 балл.*

**Постановка.** Активируйте обученный адаптер и получите ответы для тех же ID. Сохраните foundation/reference/aligned вместе с правильной буквой.

**Результат.** blind_predictions.jsonl.

**Критерий принятия.** Каждый ID имеет три фактических ответа; порядок выборок не используется вместо ID.


In [20]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/blind_predictions.jsonl для трёх политик
import json
manifest = [json.loads(x) for x in manifest_path.read_text(encoding="utf-8").splitlines() if x.strip()]
eval_records = [r for r in manifest if r["split"] == "id_test"]
all_predictions = []
for i, r in enumerate(eval_records, 1):
    # Foundation/reference — одна и та же начальная Lab 9 policy, сохранённая в reference.
    initial = generate_answer(r, "reference")
    aligned = generate_answer(r, "policy")
    all_predictions.append({"id": r["id"], "image": r["image"], "question": r["question"],
                            "correct": r["answer"], "foundation": initial, "reference": initial, "aligned": aligned})
    print(f"[{i}/{len(eval_records)}] {r['id']}: initial={initial!r}; aligned={aligned!r}")
(ARTIFACTS / "blind_predictions.jsonl").write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in all_predictions) + "\n", encoding="utf-8")
print("Сохранено:", ARTIFACTS / "blind_predictions.jsonl")

[1/64] scienceqa-test-10: initial='C'; aligned='C'
[2/64] scienceqa-test-17: initial='A'; aligned='A'
[3/64] scienceqa-test-18: initial='A'; aligned='A'
[4/64] scienceqa-test-35: initial='B'; aligned='B'
[5/64] scienceqa-test-36: initial='B'; aligned='B'
[6/64] scienceqa-test-39: initial='A'; aligned='A'
[7/64] scienceqa-test-42: initial='B'; aligned='B'
[8/64] scienceqa-test-45: initial='A'; aligned='A'
[9/64] scienceqa-test-49: initial='B'; aligned='B'
[10/64] scienceqa-test-56: initial='A'; aligned='A'
[11/64] scienceqa-test-65: initial='A'; aligned='A'
[12/64] scienceqa-test-67: initial='C'; aligned='C'
[13/64] scienceqa-test-78: initial='B'; aligned='B'
[14/64] scienceqa-test-80: initial='A'; aligned='A'
[15/64] scienceqa-test-90: initial='B'; aligned='B'
[16/64] scienceqa-test-94: initial='B'; aligned='B'
[17/64] scienceqa-test-104: initial='C'; aligned='C'
[18/64] scienceqa-test-107: initial='A'; aligned='A'
[19/64] scienceqa-test-110: initial='C'; aligned='C'
[20/64] scienceqa-

### Автоматическая и ручная оценка

*Вес: 1 балл.*

**Постановка.** Рассчитайте EM по эталонным буквам. Создайте форму ручной проверки предпочтений, галлюцинаций и формата; заполните её самостоятельно после просмотра ответов. Пропуски обозначайте pending, а не нулём.

**Результат.** objective_scores.json и blind_judgements.jsonl.

**Критерий принятия.** EM не назван человеческим win rate; неопределённые суждения не включены в ручные метрики; для финальной сдачи форма заполнена.


In [25]:
# Реализуйте это задание по постановке выше.
# Артефакт: проверенный blind_judgements.jsonl на каждый id
import json, re, time, random
from IPython.display import display
from PIL import Image as PILImage

pred_path = ARTIFACTS / "blind_predictions.jsonl"
if not pred_path.exists():
    raise FileNotFoundError(f"Не найден {pred_path}. Сначала выполни ячейку генерации ответов (blind_predictions.jsonl).")
preds = [json.loads(x) for x in pred_path.read_text(encoding="utf-8").splitlines() if x.strip()]

POLICIES = ("foundation", "reference", "aligned")
FORMAT_RE = re.compile(r"^\s*\(?\s*[A-F]\s*\)?[.):]?\s*$", re.IGNORECASE)

def extract_letter(text):
    text = str(text).strip().upper()
    m = re.match(r"^\s*\(?\s*([A-F])\s*[).:]?\s*$", text)
    if m:
        return m.group(1)
    m = re.search(r"\b(?:ANSWER\s*[:=]?\s*)?([A-F])\b", text)
    return m.group(1) if m else None

# 1) Объективная оценка: EM и соблюдение формата. Это НЕ человеческий win rate.
def objective_metric(rows, policy):
    valid = [r for r in rows if r.get("correct")]
    if not valid:
        return {"exact_match": None, "format_rate": None, "correct": 0, "n": 0}
    gold = [str(r["correct"]).strip().upper() for r in valid]
    predictions = [str(r.get(policy, "")) for r in valid]
    correct = sum(extract_letter(pred) == answer for pred, answer in zip(predictions, gold))
    formatted = sum(bool(FORMAT_RE.fullmatch(pred.strip())) for pred in predictions)
    return {"exact_match": correct / len(valid), "format_rate": formatted / len(valid),
            "correct": correct, "n": len(valid)}

objective = {
    "metric": "EM: извлечённая буква ответа равна эталонной; это объективная метрика, не человеческий win rate",
    "n": len([r for r in preds if r.get("correct")]),
    "exact_match": {p: objective_metric(preds, p)["exact_match"] for p in POLICIES},
    "format_rate": {p: objective_metric(preds, p)["format_rate"] for p in POLICIES},
    "details": {p: objective_metric(preds, p) for p in POLICIES},
}
save_json("objective_scores.json", objective)
print("Сохранён objective_scores.json:")
print(json.dumps(objective, ensure_ascii=False, indent=2))

# 2) Обычная консольная форма ручной проверки: картинка, вопрос и три ответа.
# Политики для каждого ID показываются под случайными номерами; соответствие сохраняется в JSONL.
J_PATH = ARTIFACTS / "blind_judgements.jsonl"
def load_judgements():
    if not J_PATH.exists():
        return {r["id"]: {"id": r["id"], "status": "pending"} for r in preds}
    existing = {}
    for line in J_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line); existing[row["id"]] = row
    return {r["id"]: existing.get(r["id"], {"id": r["id"], "status": "pending"}) for r in preds}

def save_judgements(rows_by_id):
    J_PATH.write_text("".join(json.dumps(rows_by_id[r["id"]], ensure_ascii=False) + "\n" for r in preds), encoding="utf-8")

def parse_numbers(answer):
    if not answer.strip():
        return []
    try:
        nums = sorted({int(x.strip()) for x in answer.split(",") if x.strip()})
    except ValueError:
        return []
    return [n for n in nums if n in (1, 2, 3)]

def run_manual_review(reviewer="reviewer"):
    rows = load_judgements()
    rng = random.Random(SEED)
    for index, p in enumerate(preds, 1):
        old = rows[p["id"]]
        if old.get("status") == "done":
            continue
        print("\n" + "=" * 78)
        print(f"Пример {index}/{len(preds)} | ID: {p['id']}")
        image_path = p.get("image")
        if image_path and Path(image_path).is_file():
            try:
                with PILImage.open(image_path) as im:
                    display(im.convert("RGB").copy())
            except Exception as e:
                print(f"Не удалось показать изображение ({e}); путь: {image_path}")
        else:
            print(f"Изображение не найдено: {image_path}")
        print("ВОПРОС:\n" + str(p.get("question", "")))
        print("ЭТАЛОННАЯ БУКВА (для объективной проверки; при слепой оценке лучше не смотреть):", p.get("correct", "?"))

        order = list(POLICIES)
        rng.shuffle(order)
        print("\nОТВЕТЫ МОДЕЛЕЙ (названия политик скрыты):")
        for number, policy in enumerate(order, 1):
            print(f"  Ответ {number}: {p.get(policy, '')!r}")

        choice = input("Лучший ответ — номера через запятую; 0 = все равнозначны; s = пропустить; q = выход: ").strip().lower()
        if choice == "q":
            break
        if choice == "s" or not choice:
            continue
        if choice == "0":
            best_nums = []
            tie_all = True
        else:
            best_nums = parse_numbers(choice)
            tie_all = False
            if not best_nums or any(x not in (1, 2, 3) for x in [int(y.strip()) for y in choice.split(",") if y.strip().isdigit()]):
                print("Ввод не распознан. Запись оставлена pending.")
                continue

        hall_input = input("Номера ответов с галлюцинацией/неподтверждённым утверждением (пусто = нет, s = пропустить запись): ").strip().lower()
        if hall_input == "s":
            continue
        hall_nums = parse_numbers(hall_input)
        fmt_input = input("Номера ответов с нарушением формата (пусто = нет): ").strip()
        bad_format_nums = parse_numbers(fmt_input)
        hall_by_policy = {policy: (number in hall_nums) for number, policy in enumerate(order, 1)}
        format_ok_by_policy = {policy: (number not in bad_format_nums) for number, policy in enumerate(order, 1)}
        best_policies = [order[n - 1] for n in best_nums]
        preferred = "tie" if tie_all or len(best_policies) != 1 else best_policies[0]
        rows[p["id"]] = {
            "id": p["id"], "status": "done", "preferred_policy": preferred,
            "best_policies": best_policies, "tie_all": tie_all,
            "blind_order": order,
            "foundation_hallucination": hall_by_policy["foundation"],
            "reference_hallucination": hall_by_policy["reference"],
            "aligned_hallucination": hall_by_policy["aligned"],
            "foundation_format_ok": format_ok_by_policy["foundation"],
            "reference_format_ok": format_ok_by_policy["reference"],
            "aligned_format_ok": format_ok_by_policy["aligned"],
            "reviewer": reviewer,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        }
        save_judgements(rows)  # сохраняем после каждой записи, чтобы не потерять прогресс
        print(f"Оценка сохранена для ID {p['id']}.")
    save_judgements(rows)
    done = sum(r.get("status") == "done" for r in rows.values())
    pending = len(rows) - done
    print(f"\nФайл: {J_PATH}\nОценено: {done}; pending: {pending}")
    return rows

rows_j = load_judgements()
save_judgements(rows_j)  # создаём файл даже если все записи пока pending
print(f"Форма ручной проверки — обычный ввод через input(). Записей: {len(preds)}.")
if any(r.get("status") == "pending" for r in rows_j.values()) and not FAST_DEV_RUN:
    reviewer = input("Имя/ID проверяющего: ").strip() or "reviewer"
    rows_j = run_manual_review(reviewer)
else:
    print("Ручная форма пропущена: FAST_DEV_RUN=True либо все записи уже оценены.")
print("Можно запускать ячейку повторно: уже оценённые записи со status='done' будут пропущены.")


Сохранён objective_scores.json:
{
  "metric": "EM: извлечённая буква ответа равна эталонной; это объективная метрика, не человеческий win rate",
  "n": 64,
  "exact_match": {
    "foundation": 0.875,
    "reference": 0.875,
    "aligned": 0.875
  },
  "format_rate": {
    "foundation": 1.0,
    "reference": 1.0,
    "aligned": 1.0
  },
  "details": {
    "foundation": {
      "exact_match": 0.875,
      "format_rate": 1.0,
      "correct": 56,
      "n": 64
    },
    "reference": {
      "exact_match": 0.875,
      "format_rate": 1.0,
      "correct": 56,
      "n": 64
    },
    "aligned": {
      "exact_match": 0.875,
      "format_rate": 1.0,
      "correct": 56,
      "n": 64
    }
  }
}
Форма ручной проверки — обычный ввод через input(). Записей: 64.
Ручная форма пропущена: FAST_DEV_RUN=True либо все записи уже оценены.
Можно запускать ячейку повторно: уже оценённые записи со status='done' будут пропущены.


### Эффект DPO

*Вес: 1 балл.*

**Постановка.** Сопоставьте EM трёх политик, а по заполненным суждениям — win rate, hallucination rate и соблюдение формата. Укажите число оценённых записей и ограничения label-derived предпочтений.

**Результат.** blind_report.json и вывод.

**Критерий принятия.** Метрики имеют реальные знаменатели; pending-оценка не выдаётся за завершённую; вывод не обобщает предпочтение правильной буквы на безопасность или универсальную полезность.


In [26]:
# Реализуйте это задание по постановке выше.
# Артефакт: blind_report.json из экспертных оценок и ответов
import json
preds = [json.loads(x) for x in (ARTIFACTS / "blind_predictions.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
judgements = [json.loads(x) for x in (ARTIFACTS / "blind_judgements.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
objective = json.loads((ARTIFACTS / "objective_scores.json").read_text(encoding="utf-8"))
valid_j = [j for j in judgements if j.get("preferred_policy") in {"foundation", "reference", "aligned", "tie"}]
win = {k: sum(j["preferred_policy"] == k for j in valid_j) / len(valid_j) if valid_j else None for k in ("foundation", "reference", "aligned", "tie")}
report = {
    "objective_em": objective,
    "human_audit": {"n_predictions": len(preds), "n_judged_preference": len(valid_j), "preference_win_rate": win,
        "hallucination_rates": {}, "format_compliance_rates": {},
        "note": "Rates are null until corresponding human fields are filled. Pending entries are excluded."},
    "limitations": ["Preference pairs are derived from ScienceQA answer labels, not human helpfulness judgments.",
                    "EM on a small operational ID subset does not establish general VLM quality.",
                    "The reference is a frozen copy of the initial Lab 9 adapter.",
                    "Human win-rate/hallucination/format metrics require completed blind judgments."]
}
for policy in ("foundation", "reference", "aligned"):
    hkey, fkey = f"{policy}_hallucination", f"{policy}_format_ok"
    h = [j[hkey] for j in judgements if j.get(hkey) in {True, False, "yes", "no"}]
    f = [j[fkey] for j in judgements if j.get(fkey) in {True, False, "yes", "no"}]
    report["human_audit"]["hallucination_rates"][policy] = {"rate": sum(v is True or v == "yes" for v in h) / len(h) if h else None, "n": len(h)}
    report["human_audit"]["format_compliance_rates"][policy] = {"rate": sum(v is True or v == "yes" for v in f) / len(f) if f else None, "n": len(f)}
save_json("blind_report.json", report)
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "objective_em": {
    "metric": "EM: извлечённая буква ответа равна эталонной; это объективная метрика, не человеческий win rate",
    "n": 64,
    "exact_match": {
      "foundation": 0.875,
      "reference": 0.875,
      "aligned": 0.875
    },
    "format_rate": {
      "foundation": 1.0,
      "reference": 1.0,
      "aligned": 1.0
    },
    "details": {
      "foundation": {
        "exact_match": 0.875,
        "format_rate": 1.0,
        "correct": 56,
        "n": 64
      },
      "reference": {
        "exact_match": 0.875,
        "format_rate": 1.0,
        "correct": 56,
        "n": 64
      },
      "aligned": {
        "exact_match": 0.875,
        "format_rate": 1.0,
        "correct": 56,
        "n": 64
      }
    }
  },
  "human_audit": {
    "n_predictions": 64,
    "n_judged_preference": 64,
    "preference_win_rate": {
      "foundation": 0.0,
      "reference": 0.0,
      "aligned": 0.0,
      "tie": 1.0
    },
    "hallucination_rates": {
      "foundati

### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [ ]:
# Реализуйте творческое мини-исследование по постановке выше.
raise NotImplementedError("Реализуйте бонусное исследование")

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.